# BST-BEV — Index Memory and Storage Analysis

This notebook measures the **absolute memory/storage cost** of the current BST indexes rather than only reporting multipliers.

It compares:

- fixed BST at **z24**
- hierarchical BST at **z19–24**

For each nuScenes map it records:

- key count
- primitive memberships
- build time
- retained / peak Python allocations with `tracemalloc`
- optional process RSS change
- recursive retained size of the actual index containers
- estimated compact packed size using 64-bit keys, 64-bit posting offsets, and 32-bit primitive IDs

The compact estimate excludes the underlying HD-map geometry because that geometry is common to both approaches.

Outputs:

- `outputs/index_memory_by_map.csv`
- `outputs/index_memory_summary.csv`


In [1]:
from pathlib import Path
import sys
import gc
import time
import tracemalloc

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

DATAROOT = PROJECT_ROOT / "data" / "nuscenes"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("nuScenes root:", DATAROOT)


Project root: /Users/abinavbala/College/Research/BST-BEV
nuScenes root: /Users/abinavbala/College/Research/BST-BEV/data/nuscenes


In [2]:
from nuscenes.nuscenes import NuScenes
from nuscenes.map_expansion.map_api import NuScenesMap

from bst_bev.maps.bst_map_index import BSTMapIndex
from bst_bev.maps.hierarchical_bst_map_index import HierarchicalBSTMapIndex

try:
    import psutil
    PSUTIL_AVAILABLE = True
except ImportError:
    PSUTIL_AVAILABLE = False

print("psutil available:", PSUTIL_AVAILABLE)


psutil available: True


In [3]:
MAP_LAYERS = (
    "lane_divider",
    "road_divider",
    "ped_crossing",
)

FIXED_ZOOM = 24
HIERARCHICAL_MIN_ZOOM = 19
HIERARCHICAL_MAX_ZOOM = 24
INDEX_SPACING_M = 1.0

# Recursive deep sizing is useful but slower.
RUN_DEEP_SIZE = True

# Compact CSR-like representation assumptions.
PACKED_KEY_BYTES = 8
PACKED_OFFSET_BYTES = 8
PACKED_PRIMITIVE_ID_BYTES = 4

BY_MAP_OUTPUT = OUTPUT_DIR / "index_memory_by_map.csv"
SUMMARY_OUTPUT = OUTPUT_DIR / "index_memory_summary.csv"


In [4]:
nusc = NuScenes(
    version="v1.0-mini",
    dataroot=str(DATAROOT),
    verbose=False,
)

locations = sorted({
    nusc.get(
        "log",
        nusc.get("scene", scene["token"])["log_token"],
    )["location"]
    for scene in nusc.scene
})

nusc_maps = {
    location: NuScenesMap(
        dataroot=str(DATAROOT),
        map_name=location,
    )
    for location in locations
}

print("Locations:")
for location in locations:
    print(" -", location)


Locations:
 - boston-seaport
 - singapore-hollandvillage
 - singapore-onenorth
 - singapore-queenstown


## Memory helpers

`tracemalloc` is the most useful measurement for the Python object representation. RSS is included when `psutil` is available, but it is noisier because Python's allocator may retain arenas.

The recursive helper below sizes only the index containers passed to it, so it does not accidentally include the already-loaded nuScenes map.


In [5]:
def bytes_to_mb(value):
    if value is None:
        return np.nan
    try:
        if np.isnan(value):
            return np.nan
    except TypeError:
        pass
    return float(value) / (1024 ** 2)


def get_process_rss_bytes():
    if not PSUTIL_AVAILABLE:
        return np.nan
    return float(psutil.Process().memory_info().rss)


def deep_sizeof_many(*objects):
    seen = set()

    def visit(obj):
        oid = id(obj)
        if oid in seen:
            return 0
        seen.add(oid)

        size = sys.getsizeof(obj)

        if isinstance(obj, dict):
            for key, value in obj.items():
                size += visit(key)
                size += visit(value)
            return size

        if isinstance(obj, (list, tuple, set, frozenset)):
            for item in obj:
                size += visit(item)
            return size

        if hasattr(obj, "__dict__"):
            size += visit(vars(obj))

        slots = getattr(type(obj), "__slots__", ())
        if isinstance(slots, str):
            slots = (slots,)

        for slot in slots:
            if slot in ("__dict__", "__weakref__"):
                continue
            try:
                value = getattr(obj, slot)
            except AttributeError:
                continue
            size += visit(value)

        return size

    return sum(visit(obj) for obj in objects)


def packed_index_bytes(*, key_count, membership_count):
    return (
        key_count * PACKED_KEY_BYTES
        + (key_count + 1) * PACKED_OFFSET_BYTES
        + membership_count * PACKED_PRIMITIVE_ID_BYTES
    )


## Measurement functions

The fixed and hierarchical indexes are built separately so their allocations can be measured independently.


In [6]:
def measure_fixed_index(*, nusc_map, location):
    gc.collect()
    rss_before = get_process_rss_bytes()
    tracemalloc.start()

    start = time.perf_counter()

    index = BSTMapIndex(
        nusc_map=nusc_map,
        location=location,
        zoom=FIXED_ZOOM,
        index_spacing_m=INDEX_SPACING_M,
        adaptive_index_spacing=False,
    )
    index.build()

    build_time_s = time.perf_counter() - start

    gc.collect()
    current_bytes, peak_bytes = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    rss_after = get_process_rss_bytes()
    rss_delta_bytes = (
        rss_after - rss_before
        if PSUTIL_AVAILABLE
        else np.nan
    )

    key_count = len(index.index)
    membership_count = sum(
        len(postings)
        for postings in index.index.values()
    )

    primitive_count = len({
        primitive
        for postings in index.index.values()
        for primitive in postings
    })

    deep_bytes = (
        deep_sizeof_many(index.index)
        if RUN_DEEP_SIZE
        else np.nan
    )

    packed_bytes = packed_index_bytes(
        key_count=key_count,
        membership_count=membership_count,
    )

    result = {
        "location": location,
        "index_type": "fixed",
        "min_zoom": FIXED_ZOOM,
        "max_zoom": FIXED_ZOOM,
        "key_count": key_count,
        "membership_count": membership_count,
        "primitive_count": primitive_count,
        "build_time_s": build_time_s,
        "tracemalloc_current_bytes": current_bytes,
        "tracemalloc_peak_bytes": peak_bytes,
        "rss_delta_bytes": rss_delta_bytes,
        "deep_size_bytes": deep_bytes,
        "packed_estimate_bytes": packed_bytes,
    }

    del index
    gc.collect()

    return result


def measure_hierarchical_index(*, nusc_map, location):
    gc.collect()
    rss_before = get_process_rss_bytes()
    tracemalloc.start()

    start = time.perf_counter()

    index = HierarchicalBSTMapIndex(
        nusc_map=nusc_map,
        location=location,
        min_zoom=HIERARCHICAL_MIN_ZOOM,
        max_zoom=HIERARCHICAL_MAX_ZOOM,
        index_spacing_m=INDEX_SPACING_M,
    )
    index.build()

    build_time_s = time.perf_counter() - start

    gc.collect()
    current_bytes, peak_bytes = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    rss_after = get_process_rss_bytes()
    rss_delta_bytes = (
        rss_after - rss_before
        if PSUTIL_AVAILABLE
        else np.nan
    )

    key_count = index.key_count
    membership_count = index.membership_count
    primitive_count = index.primitive_count

    deep_bytes = (
        deep_sizeof_many(
            index.spatial_index,
            index.primitive_store,
        )
        if RUN_DEEP_SIZE
        else np.nan
    )

    packed_bytes = packed_index_bytes(
        key_count=key_count,
        membership_count=membership_count,
    )

    result = {
        "location": location,
        "index_type": "hierarchical",
        "min_zoom": HIERARCHICAL_MIN_ZOOM,
        "max_zoom": HIERARCHICAL_MAX_ZOOM,
        "key_count": key_count,
        "membership_count": membership_count,
        "primitive_count": primitive_count,
        "build_time_s": build_time_s,
        "tracemalloc_current_bytes": current_bytes,
        "tracemalloc_peak_bytes": peak_bytes,
        "rss_delta_bytes": rss_delta_bytes,
        "deep_size_bytes": deep_bytes,
        "packed_estimate_bytes": packed_bytes,
    }

    del index
    gc.collect()

    return result


## Run over all nuScenes-mini maps


In [7]:
results = []
experiment_start = time.perf_counter()

for location in locations:
    print()
    print("=" * 72)
    print(location)
    print("=" * 72)

    nusc_map = nusc_maps[location]

    print("Building fixed index...")
    fixed_result = measure_fixed_index(
        nusc_map=nusc_map,
        location=location,
    )
    results.append(fixed_result)

    print(
        "  keys:",
        f"{fixed_result['key_count']:,}",
    )
    print(
        "  memberships:",
        f"{fixed_result['membership_count']:,}",
    )
    print(
        "  Python deep size:",
        f"{bytes_to_mb(fixed_result['deep_size_bytes']):.2f} MB",
    )
    print(
        "  packed estimate:",
        f"{bytes_to_mb(fixed_result['packed_estimate_bytes']):.2f} MB",
    )

    print()
    print("Building hierarchical index...")
    hierarchical_result = measure_hierarchical_index(
        nusc_map=nusc_map,
        location=location,
    )
    results.append(hierarchical_result)

    print(
        "  keys:",
        f"{hierarchical_result['key_count']:,}",
    )
    print(
        "  memberships:",
        f"{hierarchical_result['membership_count']:,}",
    )
    print(
        "  Python deep size:",
        f"{bytes_to_mb(hierarchical_result['deep_size_bytes']):.2f} MB",
    )
    print(
        "  packed estimate:",
        f"{bytes_to_mb(hierarchical_result['packed_estimate_bytes']):.2f} MB",
    )

results_df = pd.DataFrame(results)

print()
print(
    "Completed in",
    f"{(time.perf_counter() - experiment_start) / 60:.2f}",
    "minutes",
)



boston-seaport
Building fixed index...
  keys: 33,303
  memberships: 34,508
  Python deep size: 11.20 MB
  packed estimate: 0.64 MB

Building hierarchical index...
  keys: 297,437
  memberships: 466,330
  Python deep size: 100.46 MB
  packed estimate: 6.32 MB

singapore-hollandvillage
Building fixed index...
  keys: 7,250
  memberships: 7,327
  Python deep size: 2.45 MB
  packed estimate: 0.14 MB

Building hierarchical index...
  keys: 70,414
  memberships: 100,162
  Python deep size: 23.84 MB
  packed estimate: 1.46 MB

singapore-onenorth
Building fixed index...
  keys: 10,655
  memberships: 10,910
  Python deep size: 3.49 MB
  packed estimate: 0.20 MB

Building hierarchical index...
  keys: 103,158
  memberships: 148,255
  Python deep size: 36.29 MB
  packed estimate: 2.14 MB

singapore-queenstown
Building fixed index...
  keys: 7,759
  memberships: 7,966
  Python deep size: 2.62 MB
  packed estimate: 0.15 MB

Building hierarchical index...
  keys: 81,571
  memberships: 110,365
  Py

In [8]:
BYTE_COLUMNS = [
    "tracemalloc_current_bytes",
    "tracemalloc_peak_bytes",
    "rss_delta_bytes",
    "deep_size_bytes",
    "packed_estimate_bytes",
]

for column in BYTE_COLUMNS:
    results_df[
        column.replace("_bytes", "_mb")
    ] = results_df[column] / (1024 ** 2)

results_df[
    [
        "location",
        "index_type",
        "key_count",
        "membership_count",
        "primitive_count",
        "build_time_s",
        "tracemalloc_current_mb",
        "tracemalloc_peak_mb",
        "rss_delta_mb",
        "deep_size_mb",
        "packed_estimate_mb",
    ]
]


,location,index_type,key_count,membership_count,primitive_count,build_time_s,tracemalloc_current_mb,tracemalloc_peak_mb,rss_delta_mb,deep_size_mb,packed_estimate_mb
0,boston-seaport,fixed,33303,34508,1388,2.322634,12.013916,12.017406,10.562500,11.197871,0.639809
1,boston-seaport,hierarchical,297437,466330,1388,11.007835,100.439007,100.447468,13.312500,100.460360,6.317444
2,singapore-hollandvillage,fixed,7250,7327,355,0.677616,3.344513,3.347851,4.156250,2.453386,0.138584
3,singapore-hollandvillage,hierarchical,70414,100162,355,1.935791,24.326984,24.342892,-28.546875,23.839147,1.456528
4,singapore-onenorth,fixed,10655,10910,629,0.967260,4.357166,4.360489,3.984375,3.486206,0.204208
5,singapore-onenorth,hierarchical,103158,148255,629,3.401640,36.597804,36.610103,40.203125,36.290697,2.139622
6,singapore-queenstown,fixed,7759,7966,504,0.720376,3.500797,3.504097,3.562500,2.620658,0.148788
7,singapore-queenstown,hierarchical,81571,110365,504,2.319855,27.384678,27.390209,13.015625,26.912811,1.665691


## Aggregate absolute storage across all four maps

Do not sum RSS deltas; they are allocator/process measurements rather than clean retained-object sizes.


In [9]:
summary = (
    results_df
    .groupby("index_type")
    .agg(
        maps=("location", "count"),
        total_keys=("key_count", "sum"),
        total_memberships=("membership_count", "sum"),
        total_build_time_s=("build_time_s", "sum"),
        total_tracemalloc_current_mb=("tracemalloc_current_mb", "sum"),
        total_tracemalloc_peak_mb=("tracemalloc_peak_mb", "sum"),
        total_deep_size_mb=("deep_size_mb", "sum"),
        total_packed_estimate_mb=("packed_estimate_mb", "sum"),
    )
    .reset_index()
)

summary


,index_type,maps,total_keys,total_memberships,total_build_time_s,total_tracemalloc_current_mb,total_tracemalloc_peak_mb,total_deep_size_mb,total_packed_estimate_mb
0,fixed,4,58967,60711,4.687886,23.216393,23.229843,19.758121,1.131390
1,hierarchical,4,552580,825112,18.665121,188.748473,188.790671,187.503015,11.579285


## Fixed vs hierarchical: multipliers and absolute MB


In [10]:
fixed = summary[
    summary["index_type"] == "fixed"
].iloc[0]

hier = summary[
    summary["index_type"] == "hierarchical"
].iloc[0]

comparison = pd.DataFrame([
    {
        "metric": "keys",
        "fixed": fixed["total_keys"],
        "hierarchical": hier["total_keys"],
        "multiplier": hier["total_keys"] / fixed["total_keys"],
    },
    {
        "metric": "memberships",
        "fixed": fixed["total_memberships"],
        "hierarchical": hier["total_memberships"],
        "multiplier": (
            hier["total_memberships"]
            / fixed["total_memberships"]
        ),
    },
    {
        "metric": "Python deep size MB",
        "fixed": fixed["total_deep_size_mb"],
        "hierarchical": hier["total_deep_size_mb"],
        "multiplier": (
            hier["total_deep_size_mb"]
            / fixed["total_deep_size_mb"]
        ),
    },
    {
        "metric": "packed estimate MB",
        "fixed": fixed["total_packed_estimate_mb"],
        "hierarchical": hier["total_packed_estimate_mb"],
        "multiplier": (
            hier["total_packed_estimate_mb"]
            / fixed["total_packed_estimate_mb"]
        ),
    },
])

comparison


,metric,fixed,hierarchical,multiplier
0,keys,58967.000000,552580.000000,9.371004
1,memberships,60711.000000,825112.000000,13.590816
2,Python deep size MB,19.758121,187.503015,9.489921
3,packed estimate MB,1.131390,11.579285,10.234569


## How much of the current RAM is Python-object overhead?


In [11]:
summary["python_to_packed_ratio"] = (
    summary["total_deep_size_mb"]
    / summary["total_packed_estimate_mb"]
)

summary[
    [
        "index_type",
        "total_deep_size_mb",
        "total_packed_estimate_mb",
        "python_to_packed_ratio",
    ]
]


,index_type,total_deep_size_mb,total_packed_estimate_mb,python_to_packed_ratio
0,fixed,19.758121,1.131390,17.463587
1,hierarchical,187.503015,11.579285,16.192970


## Deployment-scale thought experiment

This is not a measurement. It simply applies the measured packed-size multiplier to hypothetical fixed-index sizes so you can judge whether the multiplier matters in absolute terms.


In [12]:
packed_multiplier = (
    hier["total_packed_estimate_mb"]
    / fixed["total_packed_estimate_mb"]
)

deployment_df = pd.DataFrame({
    "fixed_index_mb": [
        10,
        100,
        1024,
        10 * 1024,
    ]
})

deployment_df["hierarchical_index_mb"] = (
    deployment_df["fixed_index_mb"]
    * packed_multiplier
)

deployment_df["hierarchical_index_gb"] = (
    deployment_df["hierarchical_index_mb"]
    / 1024
)

deployment_df


,fixed_index_mb,hierarchical_index_mb,hierarchical_index_gb
0,10,102.345686,0.099947
1,100,1023.456861,0.999470
2,1024,10480.198255,10.234569
3,10240,104801.982555,102.345686


## Save results


In [13]:
results_df.to_csv(
    BY_MAP_OUTPUT,
    index=False,
)

summary.to_csv(
    SUMMARY_OUTPUT,
    index=False,
)

print("Saved:")
print(" -", BY_MAP_OUTPUT)
print(" -", SUMMARY_OUTPUT)


Saved:
 - /Users/abinavbala/College/Research/BST-BEV/outputs/index_memory_by_map.csv
 - /Users/abinavbala/College/Research/BST-BEV/outputs/index_memory_summary.csv


## Interpretation

The most useful numbers are:

1. **Python deep size** — approximate RAM retained by the current object-heavy prototype.
2. **Packed estimate** — approximate storage for the same inverted index represented efficiently with integer arrays.
3. **Absolute MB/GB difference** — more meaningful than the multiplier alone.

If the Python deep-size result is much larger than the packed estimate, then the current RAM cost is dominated by representation overhead (`dict`, `set`, dataclasses, Python integers), not by the intrinsic information content of the hierarchical index.

The packed estimate intentionally excludes HD-map geometry because both fixed and hierarchical retrieval ultimately need access to the same geometry.
